# Investor Lens Analyzer — Phase 4: Multi-Investor Comparison

Runs a target company through **all available investor lenses** simultaneously and produces:
- Individual verdict per investor
- Side-by-side comparison table (Deliverable 4)
- Consensus view + key disagreements + synthesis

**To switch company:** edit only the `⚙️ CONFIGURATION` cell.

## ⚙️ CONFIGURATION

In [31]:
# ============================================================
#  USER CONFIG — edit ONLY this cell
# ============================================================

# Select which investors to include (must have Phase 1 outputs on Drive)
# Seth Klarman is excluded because the original source PDF returns HTTP 403 in Colab.
# Add him back only after manually uploading a stable Phase 1 DNA/evidence output.
ACTIVE_INVESTORS = [
    "warren_buffett",
    "howard_marks",
    "peter_lynch",
    "nick_sleep",
]

INVESTOR_REGISTRY = {
    "warren_buffett": {"display_name": "Warren Buffett",  "style": "Quality Compounder"},
    "howard_marks":   {"display_name": "Howard Marks",    "style": "Credit/Cycles"},
    "seth_klarman":   {"display_name": "Seth Klarman",    "style": "Value/Distressed"},
    "peter_lynch":    {"display_name": "Peter Lynch",     "style": "Value/Growth"},
    "nick_sleep":     {"display_name": "Nick Sleep",      "style": "Value/Distressed"},
}

# ─── Target Company ───────────────────────────────────────
# We choose NVIDIA, Costco, JPMorgan Chase here to apply all investor lenses, run them one by one
COMPANY_NAME  = "NVIDIA Corporation"   # ← CHANGE
TICKER        = "NVDA"              # ← CHANGE
CURRENT_PRICE = "$211.50"           # ← CHANGE
MARKET_CAP    = "$5.14 trillion"     # ← CHANGE

TARGET_DESCRIPTION = """
NVIDIA Corporation is the leading provider of GPUs and accelerated computing
platforms used in artificial intelligence, data centers, gaming, professional
visualization, networking, and autonomous systems.

The company has become the central infrastructure supplier for the AI boom,
with strong demand for data center GPUs, CUDA software ecosystem advantages,
high margins, and powerful customer relationships with hyperscalers and AI labs.
Its moat comes from hardware performance, software lock-in, developer ecosystem,
and full-stack AI infrastructure capabilities. However, the company also faces
major questions around valuation, cyclicality in semiconductor demand, customer
concentration, competition from custom AI chips, export restrictions, and whether
current AI infrastructure spending can support long-term earnings expectations.
"""

# COMPANY_NAME  = "Costco Wholesale Corporation"
# TICKER        = "COST"
# CURRENT_PRICE = "$1012.06"
# MARKET_CAP    = "$449.00 billion"

# TARGET_DESCRIPTION = """
# Costco Wholesale Corporation operates a global membership warehouse retail model,
# selling groceries, household goods, appliances, apparel, fuel, and other products
# at low margins to a loyal membership base.

# The company's core advantage is its scale-economics-sharing model: Costco uses
# purchasing scale, operational efficiency, and low markups to offer lower prices
# to customers, which drives loyalty, membership renewal, traffic, and further
# scale. Membership fees provide recurring high-margin income, while the company’s
# culture emphasizes long-term customer value over short-term margin maximization.
# Key questions include whether the current valuation is too demanding, whether
# Costco can continue international expansion, whether membership pricing power
# remains strong, and whether the model can keep compounding despite its already
# large scale.
# """

# COMPANY_NAME  = "JPMorgan Chase & Co."
# TICKER        = "JPM"
# CURRENT_PRICE = "$306.27"
# MARKET_CAP    = "$820.65 billion"

# TARGET_DESCRIPTION = """
# JPMorgan Chase & Co. is the largest U.S. bank by assets and a diversified financial
# institution with major businesses in consumer banking, commercial banking,
# investment banking, asset and wealth management, credit cards, payments, and
# capital markets.

# The company benefits from scale, a large and relatively low-cost deposit base,
# strong brand trust, broad customer relationships, disciplined risk management,
# and diversified earnings streams. Its business model can generate strong returns
# on equity in normal environments, and its scale allows it to invest heavily in
# technology, compliance, payments infrastructure, and client service.

# However, JPMorgan is also exposed to financial-sector risks including credit
# cycles, loan losses, interest-rate changes, regulatory capital requirements,
# market volatility, deposit competition, and macroeconomic stress. The key
# investment question is whether JPMorgan's franchise quality and management
# discipline are strong enough to justify owning a leveraged financial institution
# through changing credit and rate cycles.
# """

# ──────────────────────────────────────────────────────────

from datetime import date
ANALYSIS_DATE = str(date.today())
print(f"✅ Company  : {COMPANY_NAME} ({TICKER}) @ {CURRENT_PRICE}")
print(f"   Investors: {[INVESTOR_REGISTRY[i]['display_name'] for i in ACTIVE_INVESTORS]}")

✅ Company  : NVIDIA Corporation (NVDA) @ $211.50
   Investors: ['Warren Buffett', 'Howard Marks', 'Peter Lynch', 'Nick Sleep']


## 1. Setup

In [32]:
import json, re, time
import pandas as pd
from pathlib import Path

from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

import google.auth
from google.auth.transport.requests import Request
from google import genai
from google.genai import types

PROJECT_ID = "analytics-in-practice-492209"
LOCATION   = "us-central1"
MODEL_NAME = "gemini-2.5-pro"

SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=SCOPES)
creds.refresh(Request())
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, credentials=creds)

BASE_DIR    = Path("/content/drive/MyDrive/UC-A8 Investor Lens Analyzer")
OUTPUT_DIR  = BASE_DIR / "Investor DNA"
COMPARE_DIR = BASE_DIR / "Comparison"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")

Mounted at /content/drive
✅ Setup complete


## 2. Load All Investor DNA Files

In [33]:
investor_data = {}

for inv_key in ACTIVE_INVESTORS:
    dna_path      = OUTPUT_DIR / f"{inv_key}_dna.md"
    evidence_path = OUTPUT_DIR / f"{inv_key}_evidence.json"

    if not dna_path.exists() or not evidence_path.exists():
        print(f"⚠️  Skipping {inv_key} — Phase 1 outputs not found. Run investor_dna.ipynb first.")
        continue

    with open(evidence_path) as f:
        evidence = json.load(f)

    investor_data[inv_key] = {
        "display_name": INVESTOR_REGISTRY[inv_key]["display_name"],
        "style":        INVESTOR_REGISTRY[inv_key]["style"],
        "dna":          dna_path.read_text(encoding="utf-8"),
        "evidence":     evidence
    }
    print(f"✅ Loaded {INVESTOR_REGISTRY[inv_key]['display_name']}")

print(f"\n{len(investor_data)} investors ready for comparison.")

✅ Loaded Warren Buffett
✅ Loaded Howard Marks
✅ Loaded Peter Lynch
✅ Loaded Nick Sleep

4 investors ready for comparison.


## 3. Run Quick Verdict Per Investor

In [34]:
# ============================================================
#  Robust comparison logic
#  Uses the actual Phase 1 structure:
#  evidence JSON keys include investment_philosophy, buy_criteria,
#  avoid_red_flags, risk_management, signature_patterns, notable_quotes, etc.
# ============================================================

VALID_VERDICTS = {"STRONG BUY", "INTERESTED", "WATCHLIST", "PASS", "HARD PASS"}
VALID_SCREENS = {"YES", "MAYBE", "NO"}

COMPARISON_PROMPT = """
You are applying {investor}'s investment framework ({style}) to {company} ({ticker}).

Company context:
{description}

Use ONLY the investor DNA summary and selected evidence below.

Return a JSON object with exactly these fields:
- initial_screen: one of YES, MAYBE, NO
- verdict: one of STRONG BUY, INTERESTED, WATCHLIST, PASS, HARD PASS
- key_appeal: one short sentence, max 25 words
- key_concern: one short sentence, max 25 words
- position_size_estimate: one short phrase, max 12 words
- reasoning: two short sentences, max 55 words total
- supporting_quote_id: integer ID from the quote candidates, or 0 if none fits

Important:
- Do not include the full quote text in JSON. Only return supporting_quote_id.
- Keep every string short.
- Return only the JSON object.

INVESTOR DNA SUMMARY:
{dna}

SELECTED EVIDENCE:
{evidence}

QUOTE CANDIDATES:
{quote_candidates}
"""

# JSON schema makes Gemini much less likely to return broken/incomplete JSON.
COMPARISON_SCHEMA = {
    "type": "object",
    "properties": {
        "initial_screen": {"type": "string", "enum": ["YES", "MAYBE", "NO"]},
        "verdict": {"type": "string", "enum": ["STRONG BUY", "INTERESTED", "WATCHLIST", "PASS", "HARD PASS"]},
        "key_appeal": {"type": "string"},
        "key_concern": {"type": "string"},
        "position_size_estimate": {"type": "string"},
        "reasoning": {"type": "string"},
        "supporting_quote_id": {"type": "integer"},
    },
    "required": [
        "initial_screen",
        "verdict",
        "key_appeal",
        "key_concern",
        "position_size_estimate",
        "reasoning",
        "supporting_quote_id",
    ],
}


def normalize_verdict(value) -> str:
    """Normalize model output to the 5 Phase 2 verdict labels."""
    if value is None:
        return "ERROR"

    s = str(value).strip().upper()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"[^A-Z \[\]X]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Checklist format, e.g. [X] INTERESTED
    checklist_patterns = [
        (r"\[X\]\s*HARD PASS", "HARD PASS"),
        (r"\[X\]\s*STRONG BUY", "STRONG BUY"),
        (r"\[X\]\s*INTERESTED", "INTERESTED"),
        (r"\[X\]\s*WATCHLIST", "WATCHLIST"),
        (r"\[X\]\s*WATCH LIST", "WATCHLIST"),
        (r"\[X\]\s*PASS", "PASS"),
    ]
    for pattern, verdict in checklist_patterns:
        if re.search(pattern, s):
            return verdict

    # Most specific first. HARD PASS must come before PASS.
    if "HARD PASS" in s:
        return "HARD PASS"
    if "STRONG BUY" in s:
        return "STRONG BUY"
    if "INTERESTED" in s:
        return "INTERESTED"
    if "WATCHLIST" in s or "WATCH LIST" in s:
        return "WATCHLIST"
    if "PASS" in s:
        return "PASS"

    if s in {"BUY", "LIKELY BUY", "YES", "PASS INITIAL SCREEN", "INITIAL SCREEN YES"}:
        return "INTERESTED"
    if s in {"NOT BUY", "NO BUY", "AVOID", "REJECT", "NO", "INITIAL SCREEN NO"}:
        return "PASS"
    if "BUY" in s:
        return "INTERESTED"

    return "ERROR"


def normalize_screen(value) -> str:
    s = str(value or "MAYBE").upper().strip()
    if s in VALID_SCREENS:
        return s
    if "YES" in s:
        return "YES"
    if "NO" in s:
        return "NO"
    return "MAYBE"


def clean_short_text(value, max_chars: int = 240) -> str:
    s = " ".join(str(value or "").split())
    return s[:max_chars]


def get_quote_candidates(evidence: dict, max_quotes: int = 8) -> list:
    """Use the actual Phase 1 notable_quotes list."""
    quotes = evidence.get("notable_quotes", []) if isinstance(evidence, dict) else []
    if not isinstance(quotes, list):
        return []
    return [str(q).strip() for q in quotes if str(q).strip()][:max_quotes]


def build_selected_evidence(evidence: dict, max_items_per_section: int = 4) -> str:
    """
    Build a small comparison context from the actual evidence JSON structure.
    This avoids passing the whole evidence JSON, which can make Gemini produce truncated JSON.
    """
    if not isinstance(evidence, dict):
        return str(evidence)[:1500]

    priority_sections = [
        "investment_philosophy",
        "buy_criteria",
        "avoid_red_flags",
        "risk_management",
        "sell_discipline",
        "position_sizing_portfolio",
        "signature_patterns",
        "blind_spots_mistakes",
    ]

    selected = {}
    for section in priority_sections:
        items = evidence.get(section, [])
        if isinstance(items, list):
            selected[section] = [clean_short_text(x, 220) for x in items[:max_items_per_section]]
        elif items:
            selected[section] = clean_short_text(items, 500)

    return json.dumps(selected, ensure_ascii=False, indent=2)


def format_quote_candidates(quotes: list) -> str:
    if not quotes:
        return "0. No direct quote available"
    lines = ["0. No direct quote available"]
    for i, q in enumerate(quotes, start=1):
        lines.append(f"{i}. {clean_short_text(q, 220)}")
    return "\n".join(lines)


def extract_json_object(raw: str) -> dict:
    """Parse JSON. Includes fallback extraction if Gemini adds markdown or text."""
    raw = (raw or "").strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.I)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        return json.loads(raw)
    except Exception:
        pass

    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        return json.loads(raw[start:end + 1])

    # Last-resort partial recovery for truncated JSON.
    recovered = {}
    patterns = {
        "initial_screen": r'"initial_screen"\s*:\s*"([^"]*)',
        "verdict": r'"verdict"\s*:\s*"([^"]*)',
        "key_appeal": r'"key_appeal"\s*:\s*"([^"]*)',
        "key_concern": r'"key_concern"\s*:\s*"([^"]*)',
        "position_size_estimate": r'"position_size_estimate"\s*:\s*"([^"]*)',
        "reasoning": r'"reasoning"\s*:\s*"([^"]*)',
        "supporting_quote_id": r'"supporting_quote_id"\s*:\s*(\d+)',
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, raw, flags=re.I | re.S)
        if m:
            recovered[key] = m.group(1)

    if recovered.get("verdict"):
        return recovered

    raise ValueError("No valid JSON object found")


def normalize_comparison_result(result: dict, quote_candidates: list) -> dict:
    result = dict(result)

    result["initial_screen"] = normalize_screen(result.get("initial_screen"))
    result["verdict"] = normalize_verdict(result.get("verdict"))

    if result["verdict"] not in VALID_VERDICTS:
        result["verdict"] = "ERROR"

    for key in ["key_appeal", "key_concern", "position_size_estimate", "reasoning"]:
        result[key] = clean_short_text(result.get(key), 300)

    try:
        quote_id = int(result.get("supporting_quote_id", 0))
    except Exception:
        quote_id = 0

    if 1 <= quote_id <= len(quote_candidates):
        result["supporting_quote"] = quote_candidates[quote_id - 1]
    else:
        result["supporting_quote"] = "No direct quote available"

    return result


def get_quick_verdict(inv_key: str, inv_info: dict) -> dict:
    evidence = inv_info.get("evidence", {})
    quote_candidates = get_quote_candidates(evidence, max_quotes=8)

    prompt = COMPARISON_PROMPT.format(
        investor=inv_info["display_name"],
        style=inv_info["style"],
        company=COMPANY_NAME,
        ticker=TICKER,
        description=TARGET_DESCRIPTION.strip(),
        # Use the markdown DNA as a concise framework summary.
        # The full evidence JSON is separately compressed by build_selected_evidence().
        dna=inv_info["dna"][:4500],
        evidence=build_selected_evidence(evidence, max_items_per_section=3),
        quote_candidates=format_quote_candidates(quote_candidates),
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                max_output_tokens=2048,
                response_mime_type="application/json",
                response_schema=COMPARISON_SCHEMA,
            )
        )
    except TypeError:
        # Fallback for environments where response_schema is not supported.
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                max_output_tokens=2048,
                response_mime_type="application/json",
            )
        )

    raw = response.text.strip()

    try:
        result = extract_json_object(raw)
        result = normalize_comparison_result(result, quote_candidates)
        result["investor_key"] = inv_key
        result["investor_name"] = inv_info["display_name"]
        result["style"] = inv_info["style"]
        return result
    except Exception as e:
        print(f"JSON error for {inv_key}: {e}")
        print("Raw response preview:", raw[:800])
        return {
            "investor_key": inv_key,
            "investor_name": inv_info["display_name"],
            "style": inv_info["style"],
            "initial_screen": "N/A",
            "verdict": "ERROR",
            "key_appeal": "N/A",
            "key_concern": "N/A",
            "supporting_quote": "No direct quote available",
            "position_size_estimate": "",
            "reasoning": raw[:800],
        }


verdicts = []
for inv_key, inv_info in investor_data.items():
    print(f"  Analyzing {inv_info['display_name']}...", end=" ")
    result = get_quick_verdict(inv_key, inv_info)
    verdicts.append(result)
    print(f"{result.get('verdict', 'ERROR')}")
    time.sleep(1)

errors = [v for v in verdicts if v.get("verdict") == "ERROR"]
if errors:
    print(f"\n⚠️ Completed with {len(errors)} JSON/verdict errors. Check raw response previews above.")
else:
    print("\n✅ All verdicts complete")


  Analyzing Warren Buffett... WATCHLIST
  Analyzing Howard Marks... HARD PASS
  Analyzing Peter Lynch... PASS
  Analyzing Nick Sleep... HARD PASS

✅ All verdicts complete


## 4. Comparison Table

In [35]:
print(f"\n{'═'*70}")
print(f"  COMPARISON TABLE: {COMPANY_NAME} ({TICKER})  |  {ANALYSIS_DATE}")
print(f"{'═'*70}")

table_data = []
for v in verdicts:
    table_data.append({
        "Investor":       v.get("investor_name", ""),
        "Style":          v.get("style", ""),
        "Initial Screen": v.get("initial_screen", ""),
        "Key Appeal":     v.get("key_appeal", ""),
        "Key Concern":    v.get("key_concern", ""),
        "Verdict":        v.get("verdict", ""),
        "Position Size":  v.get("position_size_estimate", ""),
    })

df = pd.DataFrame(table_data)
pd.set_option("display.max_colwidth", 60)
print(df.to_string(index=False))


══════════════════════════════════════════════════════════════════════
  COMPARISON TABLE: NVIDIA Corporation (NVDA)  |  2026-05-08
══════════════════════════════════════════════════════════════════════
      Investor              Style Initial Screen                                                                                                                               Key Appeal                                                                                                               Key Concern   Verdict                                            Position Size
Warren Buffett Quality Compounder          MAYBE A dominant competitive moat built on its integrated hardware and software ecosystem, making it the central platform for the AI industry.                                                                  An astronomical valuation and uncertainty over the long- WATCHLIST                                                         
  Howard Marks      Credit/Cycles             

## 5. Synthesis: Consensus, Disagreements & Key Insight

In [36]:
SYNTHESIS_PROMPT = f"""
You have received verdicts from {len(verdicts)} investors on {COMPANY_NAME} ({TICKER}).

Here are the verdicts:
{json.dumps(verdicts, indent=2)}

Write a synthesis section with these parts:

**CONSENSUS VIEW**
What do most investors agree on? (2-3 sentences)

**KEY DISAGREEMENTS**
Where do investors most sharply diverge? What drives those differences? (2-3 sentences)

**NON-OBVIOUS INSIGHT**
What does the multi-lens view reveal that a single analysis would miss? (2-3 sentences)

**DEVIL'S ADVOCATE**
Which investor's concern is most likely underappreciated by the market? Why?

Keep it concise and sharp. Output clean markdown.
"""

synth_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=SYNTHESIS_PROMPT,
    config=types.GenerateContentConfig(temperature=0.3, max_output_tokens=1500)
)
synthesis_text = synth_response.text
print(synthesis_text)

### Synthesis

**CONSENSUS VIEW**
Investors unanimously agree that NVIDIA is a phenomenal business with a dominant competitive moat built on its integrated hardware and software ecosystem. However, they also share a strong consensus that the stock's current valuation is extremely high, representing the primary concern and barrier to


## 6. Save Full Comparison Report

In [37]:
report_lines = [
    f"# Multi-Investor Comparison: {COMPANY_NAME} ({TICKER})",
    f"**Analysis Date:** {ANALYSIS_DATE}",
    f"**Current Price:** {CURRENT_PRICE}  |  **Market Cap:** {MARKET_CAP}",
    "",
    "---",
    "",
    "## Company Overview",
    TARGET_DESCRIPTION.strip(),
    "",
    "---",
    "",
    "## Investor Verdicts",
    "",
    "| Investor | Style | Initial Screen | Key Appeal | Key Concern | Verdict |",
    "| --- | --- | --- | --- | --- | --- |",
]

for v in verdicts:
    report_lines.append(
        f"| {v.get('investor_name','')} | {v.get('style','')} | {v.get('initial_screen','')} "
        f"| {v.get('key_appeal','')[:60]} | {v.get('key_concern','')[:60]} | {v.get('verdict','')} |"
    )

report_lines += [
    "",
    "---",
    "",
    "## Individual Reasoning",
    "",
]

for v in verdicts:
    report_lines += [
        f"### {v.get('investor_name','')} ({v.get('style','')})",
        f"**Verdict:** {v.get('verdict','')}  |  **Position Size:** {v.get('position_size_estimate','')}",
        "",
        v.get("reasoning", ""),
        "",
        f"> *\"{v.get('supporting_quote','')}\"*",
        "",
    ]

report_lines += [
    "---",
    "",
    "## Synthesis",
    "",
    synthesis_text,
]

report_md   = "\n".join(report_lines)
report_path = COMPARE_DIR / f"{TICKER.lower()}_comparison_{ANALYSIS_DATE}.md"
report_path.write_text(report_md, encoding="utf-8")

df.to_csv(COMPARE_DIR / f"{TICKER.lower()}_verdicts_{ANALYSIS_DATE}.csv", index=False)

print(f"✅ Saved comparison report : {report_path.name}")
print(f"✅ Saved verdicts CSV      : {TICKER.lower()}_verdicts_{ANALYSIS_DATE}.csv")

✅ Saved comparison report : nvda_comparison_2026-05-08.md
✅ Saved verdicts CSV      : nvda_verdicts_2026-05-08.csv
